Imports & Load Clean Data

In [3]:
import pandas as pd
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
import time

# Load cleaned books dataset
df_books = pd.read_csv("../data/books_cleaned.csv")
print(f" Successfully loaded {len(df_books)} books for Collaborative Filtering!")

 Successfully loaded 50000 books for Collaborative Filtering!


Simulating User-Item Interaction Matrix

In [4]:
# We simulate realistic reader interactions across top 500 popular books for 1,000 active readers
np.random.seed(42)

n_users = 1000
top_books = df_books.head(500).copy()
n_books = len(top_books)

print(f"Constructing User-Item Matrix: {n_users} Readers × {n_books} Books...")

# Initialize sparse rating matrix (most users rate 15-30 books they read)
ratings_matrix = np.zeros((n_users, n_books))

for user_id in range(n_users):
    num_rated = np.random.randint(15, 30)
    rated_book_indices = np.random.choice(n_books, size=num_rated, replace=False)
    for b_idx in rated_book_indices:
        # Readers tend to rate books they finish with 3, 4, or 5 stars
        ratings_matrix[user_id, b_idx] = np.random.choice([3.0, 4.0, 5.0], p=[0.2, 0.4, 0.4])

sparsity = (1.0 - (np.count_nonzero(ratings_matrix) / float(ratings_matrix.size))) * 100
print(f" Matrix Dimensions: {ratings_matrix.shape}")
print(f" Matrix Sparsity: {sparsity:.2f}% (Industry-standard sparse interaction matrix)")

Constructing User-Item Matrix: 1000 Readers × 500 Books...
 Matrix Dimensions: (1000, 500)
 Matrix Sparsity: 95.61% (Industry-standard sparse interaction matrix)


Matrix Factorization via TruncatedSVD

In [5]:
print(" Training Matrix Factorization (TruncatedSVD) to discover latent taste factors...")
start_time = time.time()

# We decompose the user-item matrix into 20 latent taste components
n_components = 20
svd = TruncatedSVD(n_components=n_components, random_state=42)

# Transpose to get Book Representation in Latent Taste Space
book_latent_features = svd.fit_transform(ratings_matrix.T)

elapsed = time.time() - start_time
print(f" SVD Matrix Factorization complete in {elapsed:.4f} seconds!")
print(f" Book Latent Feature Matrix Shape: {book_latent_features.shape} (500 Books × 20 Taste Factors)")
print(f" Explained Variance Ratio: {svd.explained_variance_ratio_.sum():.4f}")

 Training Matrix Factorization (TruncatedSVD) to discover latent taste factors...
 SVD Matrix Factorization complete in 1.4050 seconds!
 Book Latent Feature Matrix Shape: (500, 20) (500 Books × 20 Taste Factors)
 Explained Variance Ratio: 0.0994


Item-Item Behavioral Similarity Function

In [6]:
# Compute pairwise cosine similarity between books in latent taste space
behavioral_sim_matrix = cosine_similarity(book_latent_features)

def recommend_by_user_behavior(book_title, top_k=5):
    # Find matching book
    matches = top_books[top_books['title'].str.contains(book_title, case=False, na=False)]
    if len(matches) == 0:
        return f"Book '{book_title}' not found in top interaction catalog."

    book_idx = matches.index[0]
    matched_book_title = top_books.iloc[book_idx]['title']

    # Get similarity scores for this book with all other books
    sim_scores = behavioral_sim_matrix[book_idx]

    # Sort indices by highest similarity (ignoring the book itself at index 0)
    similar_indices = sim_scores.argsort()[::-1]
    similar_indices = [i for i in similar_indices if i != book_idx][:top_k]

    results = []
    for rank, idx in enumerate(similar_indices, 1):
        row = top_books.iloc[idx].to_dict()
        row['behavioral_similarity'] = round(float(sim_scores[idx]), 4)
        results.append(row)

    res_df = pd.DataFrame(results)
    print(f"Readers who enjoyed '{matched_book_title}' also loved:")
    return res_df[['title', 'authors', 'genres', 'average_rating', 'behavioral_similarity']]

# Test recommendations for a popular book
recommend_by_user_behavior("The Hunger Games", top_k=5)

Readers who enjoyed 'The Hunger Games (The Hunger Games, #1)' also loved:


,title,authors,genres,average_rating,behavioral_similarity
0,The Glass Castle,Jeannette Walls,General Fiction,4.24,0.8052
1,Neverwhere,Neil Gaiman,Technology & AI Science,4.18,0.7870
2,An Abundance of Katherines,John Green,General Fiction,3.63,0.7206
3,Gone Girl,Gillian Flynn,Mystery & Crime Thriller,4.03,0.6703
4,Running with Scissors,Augusten Burroughs,General Fiction,3.70,0.6553
